# Demonstrate the script `extract_mismatch_positions_from_blast_alignment.py`

Script will take an alignment from BLAST as input. And output a list 

Only really useful for close strain differences.  
**IMPORTANTLY: it is SIMPLISTIC code and not meant to handle gaps/insertions or Deletions!**

The idea is then you can use the list of strain differences as input for Exclude Regions entry in IDT's Primer Quest Tool.

This demo doesn't require much of anything special and so you should be able to use this anywhere you have Jupyter working.  
**Even with JupyterLite, it will work!**  

Two places you can try those notebook and associated script now **without needing to install anything other than a modern browser on your own computer**:   
If you need a Jupyter session with a tpyical kernel for running this notebook, go to [here](https://github.com/jupyterlab/jupyterlab#jupyterlab) and click the '`launch binder`' badge.  
When the session comes up, you can drag in a downloaded copy of this notebook and you should be good to run the entire notebook. 

For JupyterLite, you can go [here](https://jupyterlite.readthedocs.io/en/stable/) and click the '`try lite now`' badge or go to [the Try Jupyter page](https://jupyter.org/try) and click either of the first two tiles in the upper left for JupyterLab or Jupyter Notebook. (JupyterLab makes it easier because then you can just drag-and-drop the downloaded notebook to get in the session file list.) You may need to choose the '`Python (Pyodide)`' kernel when you double-click on the notebook in the session and want to run it.

----------------

Example will start with an alignment of the DNA encoding second exon of HCMV UL36 from the TB40/E and Towne strains.  
BLAST was run using the 'Align two or more sequences' option toggled to make this alignment. Specifically the setting 'Somewhat similar sequences (blastn)' was used under 'Program Selection'.

When you update this notebook to handle your input, replace the following alignment with your own obtained from BLAST. (I would suggest running this entirely with the supplied data so you can first see if it works and what type of output it should generate.)

In [1]:
%%writefile BLAST_result_TB40E_UL36_vs_Towne_strain.txt
Query  1      GTGCTTACGTCTGCTGTCAGGAGTACCTGCACCCCTTTGGTTTCGTCGAGGGTCCGGGCT  60
              |||||||||||||||||||||| ||||||||||||||||| |||||||||||||||||||
Sbjct  81681  GTGCTTACGTCTGCTGTCAGGAATACCTGCACCCCTTTGGCTTCGTCGAGGGTCCGGGCT  81622

Query  61     TTATGCGTTACCAACTCATCGTGCTCATCGGGCAGCGCGGTGGCATCTACTGCTACGATG  120
              ||||||| |||||||||||||||||||||||||||||||| |||||||||||||||||||
Sbjct  81621  TTATGCGCTACCAACTCATCGTGCTCATCGGGCAGCGCGGCGGCATCTACTGCTACGATG  81562

Query  121    ACCTGCGCGACTGCGTCTACGAGCTGGCGGCCACCATGAAGGACTTTCTGCGGCATGGCT  180
              | ||||| ||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  81561  ATCTGCGTGACTGCGTCTACGAGCTGGCGGCCACCATGAAGGACTTTCTGCGGCATGGCT  81502

Query  181    TTCGTCACTGCGACCACTTCCACACTATGCGCGACTACCAGCGGCCCATGGTGCAGTACG  240
              ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  81501  TTCGTCACTGCGACCACTTCCACACTATGCGCGACTACCAGCGGCCCATGGTGCAGTACG  81442

Query  241    ACGACTACTGGAACGCCGTCATGCTCTACCGCGGTGACGTGGAGAGCCTGTCGGCCGAGG  300
              ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  81441  ACGACTACTGGAACGCCGTCATGCTCTACCGCGGTGACGTGGAGAGCCTGTCGGCCGAGG  81382

Query  301    TGACCAAGCGGGGCTACGCTTCCTACAGCATCGACGACCCTTTCGATGAGTGTCCCGACA  360
              ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  81381  TGACCAAGCGGGGCTACGCTTCCTACAGCATCGACGACCCTTTCGATGAGTGTCCCGACA  81322

Query  361    CCCATTTCGCCTTCTGGACCCACAACACAGAGGTTATGAAGTTCAAGGAAACATCCTTCA  420
              ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  81321  CCCATTTCGCCTTCTGGACCCACAACACAGAGGTTATGAAGTTCAAGGAAACATCCTTCA  81262

Query  421    GCGTCGTCCGCGCGGGCGGCAGCATCCAGACCATGGAGCTCATGATCCGTACCGTGCCGC  480
              |||| |||||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  81261  GCGTTGTCCGCGCGGGCGGCAGCATCCAGACCATGGAGCTCATGATCCGTACCGTGCCGC  81202

Query  481    GCATCACCTGTTACCACCAGTTGTTGGGCGCGCTGGGCCACGAGGTGCCAGAACGCAAAG  540
              ||||||||||||| |||||| |||||||||||||||||||||||||||||||||||||||
Sbjct  81201  GCATCACCTGTTATCACCAGCTGTTGGGCGCGCTGGGCCACGAGGTGCCAGAACGCAAAG  81142

Query  541    AGTTCCTCGTACGCCAGTACGTCTTGGTGGACACTTTCGGCGTCGTCTACGGCTACGACC  600
              |||| |||||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  81141  AGTTTCTCGTACGCCAGTACGTCTTGGTGGACACTTTCGGCGTCGTCTACGGCTACGACC  81082

Query  601    CCGCCATGGACGCCGTCTACCGTTTGGCCGAGGACGTGGTTATGTTCACCTGCGTCATGG  660
              ||||||||||||||||||||||| ||||||||||||||||||||||||||||||||||||
Sbjct  81081  CCGCCATGGACGCCGTCTACCGTCTGGCCGAGGACGTGGTTATGTTCACCTGCGTCATGG  81022

Query  661    GAAAGAAGGGACACCGAAACCACCGTTTCAGCGGCCGGCGTGAGGCCATCGTGCGTCTGG  720
              |||||||||||||||||||||||||||||||| |||||||||||||||||||||||||||
Sbjct  81021  GAAAGAAGGGACACCGAAACCACCGTTTCAGCAGCCGGCGTGAGGCCATCGTGCGTCTGG  80962

Query  721    AAAAGACACCCACCTGTCAACATCCCaaaaaaaCCCCTGACCCCATGATCATGTTTGACG  780
              ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  80961  AAAAGACACCCACCTGTCAACATCCCAAAAAAACCCCTGACCCCATGATCATGTTTGACG  80902

Query  781    AGGATGACGACGACGAGCTGTCGCTGCCTCGAAACGTGATGACGCACGAGGAGGCCGAGA  840
              ||||||||||||||||||||||||||||||| ||||||||||||||||||||||| ||||
Sbjct  80901  AGGATGACGACGACGAGCTGTCGCTGCCTCGGAACGTGATGACGCACGAGGAGGCTGAGA  80842

Query  841    GTCGCCTGTACGACGCCATCACCGAGAACCTGATGCACTGCGTCAAACTGGTGACGACGG  900
              ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  80841  GTCGCCTGTACGACGCCATCACCGAGAACCTGATGCACTGCGTCAAACTGGTGACGACGG  80782

Query  901    ACTCGCCCCTGGCCACGCACCTGTGGCCGCAGGAGCTTCAGGCCCTGTGTGACAGCCCAG  960
              ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  80781  ACTCGCCCCTGGCCACGCACCTGTGGCCGCAGGAGCTTCAGGCCCTGTGTGACAGCCCAG  80722

Query  961    CACTGTCGCTGTGCACCGATGACGTGGAGGGTGTTCGCCAGAAGCTGCGAGCTCGGACGG  1020
              ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  80721  CACTGTCGCTGTGCACCGATGACGTGGAGGGTGTTCGCCAGAAGCTGCGAGCTCGGACGG  80662

Query  1021   GCTCGTTGCACCACTTTGAACTCTCCTACCGCTTTCACGACGAGGACCCCGAAACTTACA  1080
              |||| ||||||||| |||||||||| |||||||||||||||||||||||||| || || |
Sbjct  80661  GCTCATTGCACCACCTTGAACTCTCTTACCGCTTTCACGACGAGGACCCCGAGACCTATA  80602

Query  1081   TGGGCTTCCTCTGGGATATCCCGTCCTGCGACCGCTGCGTGCGCCGACGGCGTTTCAAGG  1140
              ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
Sbjct  80601  TGGGCTTCCTCTGGGATATCCCGTCCTGCGACCGCTGCGTGCGCCGACGGCGTTTCAAGG  80542

Query  1141   TGTGCGACGTCGGTCGACGGCACATTATTCCTGGCGCCGCCAACGGCATGCCGCCCCTCA  1200
              ||||||||| ||||||||| ||||||||||| ||||||||||||||||||||||||||||
Sbjct  80541  TGTGCGACGCCGGTCGACGACACATTATTCCCGGCGCCGCCAACGGCATGCCGCCCCTCA  80482

Query  1201   CCCCGCCACACGCCTACATGAATAACTGA  1229
              |||||||||||||||||||||| ||||||
Sbjct  80481  CCCCGCCACACGCCTACATGAACAACTGA  80453


Overwriting BLAST_result_TB40E_UL36_vs_Towne_strain.txt


We need the script `extract_mismatch_positions_from_blast_alignment.py` that will get the positions of the mismatches.

Note that if you were using a typical full Python kernerl, you could simply use the following to do that:

```shell
!curl -OL https://raw.githubusercontent.com/fomightez/sequencework/refs/heads/master/alignment-utilities/extract_mismatch_positions_from_blast_alignment.py
```

However, this notebook has been written to work in the most places and so the next cell uses a way that will work in both typical Python kernels and JupyterLite.

In [2]:
import requests
def fetch_github_file_using_requests(the_url):
    """
    Take a url for a raw GitHub file and return the file content using GitHub's CORS-enabled REST API endpoint
    """
    response = requests.get(the_url)
    response.raise_for_status()  # Raise an exception for non-200 status codes
    return response.text
url="https://raw.githubusercontent.com/fomightez/sequencework/refs/heads/master/alignment-utilities/extract_mismatch_positions_from_blast_alignment.py"
file_content_string = fetch_github_file_using_requests(url)
with open(url.rsplit("/", 1)[-1], 'w', encoding='utf-8') as output_file:
    output_file.write(file_content_string)

--------

With preparation steps complete, we are read to run the script, indicating the input alignment to use.

In [3]:
%run extract_mismatch_positions_from_blast_alignment.py BLAST_result_TB40E_UL36_vs_Towne_strain.txt --output mismatches.txt


24 mismatch(es) found (positions are 1-based, in query coordinates):

Position (1-based, query):
  23
  41
  68
  101
  122
  128
  425
  494
  501
  545
  624
  693
  812
  836
  1025
  1035
  1046
  1073
  1076
  1079
  1150
  1160
  1172
  1223

Comma-separated list of mismatch positions:
23, 41, 68, 101, 122, 128, 425, 494, 501, 545, 624, 693, 812, 836, 1025, 1035, 1046, 1073, 1076, 1079, 1150, 1160, 1172, 1223




Mismatch positions also saved to: 'mismatches.txt'



If you were doing that step in a terminal or console, it would likely be `python extract_mismatch_positions_from_blast_alignment.py`, or equivalent.

Let's see the output here by running the next cell:

In [4]:
with open("mismatches.txt") as f:
    print(f.read())


24 mismatch(es) found (positions are 1-based, in query coordinates):

Position (1-based, query):
  23
  41
  68
  101
  122
  128
  425
  494
  501
  545
  624
  693
  812
  836
  1025
  1035
  1046
  1073
  1076
  1079
  1150
  1160
  1172
  1223

Comma-separated list of mismatch positions:
23, 41, 68, 101, 122, 128, 425, 494, 501, 545, 624, 693, 812, 836, 1025, 1035, 1046, 1073, 1076, 1079, 1150, 1160, 1172, 1223



However, the input that IDT's Primer Quest Tool needs to be ranges for 'Excluded Region List' and 'Excluded Region List - Probe' under 'Custom Target Region : Regions Considered' to indicate relative positions in the TB40/E sequence that don't match exactly to Towne strain so that those will be avoided in both the primers and probe suggestions by Primer Quest Tool.

We can take that final line and make it ranges a couple of ways:

One would be to simply copy the last line above and place `positions = [` in front of it and `]` at the end and to make code to assign it as a list & then use code that takes the list items and repeats them as a range, like this:

In [5]:
positions = [23, 41, 68, 101, 122, 128, 425, 494, 501, 545, 624, 693, 812, 836, 1025, 1035, 1046, 1073, 1076, 1079, 1150, 1160, 1172, 1223]
print(",".join(f"{p}-{p}" for p in positions))

23-23,41-41,68-68,101-101,122-122,128-128,425-425,494-494,501-501,545-545,624-624,693-693,812-812,836-836,1025-1025,1035-1035,1046-1046,1073-1073,1076-1076,1079-1079,1150-1150,1160-1160,1172-1172,1223-1223


A better way would be to just grab the last line and convert it on the fly to items and then arrange those as range text. This is better because whenever you copy-paste you introduce a chance you'll make an error. So this would do similar with less human involvement:

In [6]:
with open("mismatches.txt") as f:
    positions = f.readlines()[-1].strip()
print(",".join(f"{p.strip()}-{p.strip()}" for p in positions.split(",")))

23-23,41-41,68-68,101-101,122-122,128-128,425-425,494-494,501-501,545-545,624-624,693-693,812-812,836-836,1025-1025,1035-1035,1046-1046,1073-1073,1076-1076,1079-1079,1150-1150,1160-1160,1172-1172,1223-1223


The produced text can be pasted in to the entry area on the Primer Quest Tool where you supply ranges for 'Excluded Region List' and 'Excluded Region List - Probe' under 'Custom Target Region : Regions Considered' to indicate relative positions in the TB40/E sequence that don't match exactly to Towne strain so that those will be avoided in both the primers and probe suggestions by Primer Quest Tool.

Now that you see how this works, you can replace the alignment with your own alignment and make your own lists for use with IDT's Primer Quest Tool.

-----

Enjoy!